# View a Scored Recording from Direct File Paths

Visualise raw EEG/EMG traces alongside Somnotate state predictions for one
recording, using an EDF path and a predictions parquet path you provide
directly — no `--subject`/`--date` lookup through the raw-data root's
session-folder layout.

Use this when the files you want to inspect aren't (yet) sitting in the
pipeline's expected directory structure: a recording still in a scratch
location, an experimental scoring run, or a file copied in from elsewhere.
For a normal scored session that already lives under the configured
`rawdata`/`derivatives` roots, use
`hypnose_eeg/review/viewer.py` with `--subject` instead —
it resolves both paths for you.

**`EDF_PATH` and `PARQUET_PATH` must come from the same scoring run** — the
predictions parquet is only valid against the exact EDF it was generated
from (same duration, same per-second timeline). There is no way to verify
that from the files alone, so a mismatched pair will load without error but
plot every state and every `gap`/`too_short` region in the wrong place. If
`EDF_PATH` was re-concatenated (for example, after a change to gap handling)
after `PARQUET_PATH` was produced, re-run scoring on the new EDF before
viewing it here.


## 1. Imports

This reuses the same loading/plotting helpers as the CLI viewer
(`hypnose_eeg/review/viewer.py`) so behaviour stays
identical — only path resolution differs.

Install missing packages in the notebook environment if needed:

```bash
pip install mne pandas pyedflib six
```

The viewer opens an interactive Qt window (arrow-key navigation through the
recording), so this notebook needs a graphical display — a local desktop
session, or SSH with X11 forwarding (see `docs/remote_visualization.md`).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from hypnose_somnotate.config import DEFAULT_CHANNEL_LABELS
from hypnose_somnotate.io.loading import load_somnotate_vector
from hypnose_somnotate.somnotate_pipeline.utils import configuration
from hypnose_somnotate.visualization import plot_detailed_comparison

from hypnose_eeg.io.repository_paths import get_derivatives_root, get_rawdata_root
from hypnose_eeg.review.viewer import (
    _artifact_regions,
    _downsample_signals,
    _find_downsampled_fif,
    _label_artifacts,
    _open_signal_source,
    _require_graphical_display,
)

import os

_require_graphical_display(os.environ)

%matplotlib qt


## 2. Paths & Settings

Set `EDF_PATH` and `PARQUET_PATH` directly. `ARTIFACT_PATH` is optional —
leave it `None` to skip artifact shading.

- `HOURS`: elapsed-hour window from the start of the EDF, for example
  `(3.0, 6.0)`. Leave `None` to load the complete recording.
- `EEG_CHANNEL`: `0` (EEG1) or `1` (EEG2).
- `DISPLAY_RATE_HZ`: downsample signals before plotting (for example `128`).
  Leave `None` to plot at whatever rate is actually loaded (see `FIF_PATH`
  below) rather than forcing a resample.
- `FIF_PATH`: a FIF written by `hypnose_eeg/preprocessing/downsample_recordings.py`
  to read instead of `EDF_PATH`. Leave `None` (the default) to auto-detect one
  under `DERIVATIVES_ROOT` — the recording's session folder is derived from
  `EDF_PATH`'s position under `RAWDATA_ROOT`, so auto-detection only finds
  anything for recordings that still live under the configured raw-data root.
  **A FIF is used automatically whenever one is available** — reading it is
  much faster than reading the full-rate EDF — at its own rate when
  `DISPLAY_RATE_HZ` is `None`, or at the lowest available rate that still
  meets `DISPLAY_RATE_HZ` otherwise (a FIF can't be upsampled back, so a
  `DISPLAY_RATE_HZ` higher than any available FIF falls back to the EDF). Set
  `FIF_PATH` explicitly to use one that lives outside `DERIVATIVES_ROOT` (a
  scratch downsample run, for example).
- `SHOW_SCORING_KIND`: shade `gap`/`too_short` spans that the predictions
  parquet's own `kind` column designates as not-real-signal (concatenation
  gaps, dropouts, segments too short to score) — see section 6.


In [ ]:
EDF_PATH = Path("/mnt/harris/hypnose/hypnose_eeg/rawdata/sub-063_id-430/ses-001_date-20260715/ephys/sub-063_ses-001_recording-concat.edf")
PARQUET_PATH = Path("/mnt/harris/hypnose/hypnose_eeg/testing/sub-063_ses-001_recording-concat_normglobal_somnotate_predictions.parquet")
ARTIFACT_PATH = None  # e.g. Path("/path/to/recording_artifact_epochs.parquet")
FIF_PATH = None  # e.g. Path("/path/to/recording_resampled-128hz_raw.fif"); None = auto-detect

HOURS = 0, 7  # e.g. (3.0, 6.0)
EEG_CHANNEL = 0
VIEW_LENGTH_S = 600
DISPLAY_RATE_HZ = 128
SHOW_SCORING_KIND = True

RAWDATA_ROOT = get_rawdata_root()
DERIVATIVES_ROOT = get_derivatives_root()

if not EDF_PATH.is_file():
    raise FileNotFoundError(f"EDF not found: {EDF_PATH}")
if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"Predictions parquet not found: {PARQUET_PATH}")
if ARTIFACT_PATH is not None and not Path(ARTIFACT_PATH).is_file():
    raise FileNotFoundError(f"Artifact parquet not found: {ARTIFACT_PATH}")
if FIF_PATH is not None and not Path(FIF_PATH).is_file():
    raise FileNotFoundError(f"FIF not found: {FIF_PATH}")
if EEG_CHANNEL not in (0, 1):
    raise ValueError("EEG_CHANNEL must be 0 (EEG1) or 1 (EEG2)")


## 3. Select Channels & Load Raw Signals

Loads from `FIF_PATH` (explicit or auto-detected) when `DISPLAY_RATE_HZ` is
set and a FIF at a sufficient rate is available, otherwise from `EDF_PATH`.
Both paths match channel labels the same way the CLI viewer does
(`_match_channel_labels`) and read only the requested elapsed-time window.


In [ ]:
# With no explicit DISPLAY_RATE_HZ, any pre-downsampled FIF is still faster to
# read than the full-rate EDF, so use one at its own (lowest available) rate;
# a DISPLAY_RATE_HZ only accepts a FIF that can satisfy it without upsampling.
signal_path = EDF_PATH
fif_path = FIF_PATH or _find_downsampled_fif(
    EDF_PATH, RAWDATA_ROOT, DERIVATIVES_ROOT, min_rate_hz=DISPLAY_RATE_HZ or 0.0
)
if fif_path is not None:
    signal_path = Path(fif_path)
    print(f"Using pre-downsampled {signal_path.name} instead of the full-rate EDF")

with _open_signal_source(signal_path, list(DEFAULT_CHANNEL_LABELS)) as source:
    sampling_rate_hz = source.sampling_rate_hz
    total_samples = source.total_samples

    if HOURS is not None:
        start_hour, end_hour = HOURS
        start_s, end_s = start_hour * 3600.0, end_hour * 3600.0
        range_description = f"hours {start_hour:g}-{end_hour:g}"
    else:
        start_s, end_s = 0.0, total_samples / sampling_rate_hz
        range_description = "the complete recording"

    first_sample = int(start_s * sampling_rate_hz)
    last_sample = min(int(end_s * sampling_rate_hz), total_samples)
    if first_sample >= total_samples:
        duration_h = total_samples / sampling_rate_hz / 3600.0
        raise ValueError(f"selected START is beyond the {duration_h:.3g}-hour recording")

    raw_signals = source.read_window(first_sample, last_sample)

print(
    f"Loaded {range_description} from {signal_path.name} "
    f"({len(raw_signals):,} samples at {sampling_rate_hz:g} Hz)"
)


## 4. Optional Downsampling

Skipped automatically when `DISPLAY_RATE_HZ` is `None` or already matches the
EDF's native rate.


In [ ]:
plotted_rate_hz = sampling_rate_hz
if DISPLAY_RATE_HZ is not None:
    raw_signals = _downsample_signals(raw_signals, sampling_rate_hz, DISPLAY_RATE_HZ)
    plotted_rate_hz = DISPLAY_RATE_HZ


## 5. Load Somnotate Predictions

Predictions are one label per epoch (epoch length =
`configuration.time_resolution`); slice to the same window loaded above.

Also keep the full table (not just the label vector): its `kind` column
designates each epoch as `signal`, `gap` (no real recording — for example a
concatenation gap, see `hypnose_eeg/preprocessing/concatenate_recordings.py`), or
`too_short` (a real signal segment too short to score), and section 6 shades
those spans on the plot so the displayed trace matches what the parquet
itself designates for that time — see
`hypnose_somnotate.preprocessing.gap_correction.prepare_recording`.

**Pairing check:** `PARQUET_PATH` is only valid for the EDF it was generated
from. The cell below compares the EDF's total duration against the parquet's
row count (independent of whatever `HOURS` window is loaded) and raises
immediately on a mismatch — a shifted/duplicated recording, a parquet scored
before a re-concatenation, or a parquet for the wrong session will fail here
with a clear message rather than plotting every state at the wrong
timepoint.


In [ ]:
epoch_s = float(configuration.time_resolution)

# Pairing check: compare the EDF's *full* duration against the parquet's *full*
# row count, independent of the HOURS window above. A duration mismatch means
# PARQUET_PATH doesn't score this exact EDF -- every downstream index would be
# wrong -- so fail loudly here instead of silently plotting misaligned states.
predictions_table = pd.read_parquet(PARQUET_PATH, columns=["time_s", "kind"])
parquet_epoch_count = len(predictions_table)
edf_duration_s = total_samples / sampling_rate_hz
expected_epoch_count = int(round(edf_duration_s / epoch_s))

if abs(parquet_epoch_count - expected_epoch_count) > 1:
    raise ValueError(
        f"EDF/parquet duration mismatch: {EDF_PATH.name} is {edf_duration_s:.1f}s "
        f"({expected_epoch_count} epochs at {epoch_s:g}s/epoch) but "
        f"{PARQUET_PATH.name} has {parquet_epoch_count} rows. PARQUET_PATH must be "
        "the predictions scored from this exact EDF -- re-run scoring on this file "
        "(for example after a re-concatenation) or point PARQUET_PATH at the correct "
        "predictions parquet."
    )

somnotate_vec = load_somnotate_vector(PARQUET_PATH)

first_epoch = int(start_s / epoch_s)
loaded_end_s = last_sample / sampling_rate_hz
last_epoch = min(int(loaded_end_s / epoch_s), len(somnotate_vec))
somnotate_vec = somnotate_vec[first_epoch:last_epoch]

if len(somnotate_vec) == 0:
    raise ValueError(f"No scoring predictions overlap {range_description}")


## 6. Plot

Opens the interactive viewer window. Use the arrow keys to scroll through
`VIEW_LENGTH_S`-wide windows.

`gap` spans (dim gray) and `too_short` spans (orange) come straight from the
predictions parquet's own `kind` column — this is the "parquet designation"
the state colouring alone doesn't show, since a `gap`/`too_short` epoch still
carries a `label_model` value (typically `undefined`) like any other epoch.
`ARTIFACT_PATH`, if set, adds a separate red shading from a QC artifacts
parquet.


In [ ]:
def _shade_regions(fig, regions, *, color, label):
    """Generalises view_scored_recording._label_artifacts to a chosen colour/label."""
    if not regions or not fig.axes:
        return
    for axis in fig.axes:
        for start_s, end_s in regions:
            axis.axvspan(start_s, end_s, color=color, alpha=0.16, zorder=0)
    signal_axis = fig.axes[0]
    for start_s, end_s in regions:
        signal_axis.text(
            (start_s + end_s) / 2.0,
            0.98,
            label,
            transform=signal_axis.get_xaxis_transform(),
            color=color,
            fontsize=8,
            ha="center",
            va="top",
            rotation=90,
        )


fig, _viewer = plot_detailed_comparison(
    raw_signals,
    sampling_rate_hz=plotted_rate_hz,
    somnotate_vec=somnotate_vec,
    eeg_channel=EEG_CHANNEL,
    view_length_s=VIEW_LENGTH_S,
)

if SHOW_SCORING_KIND:
    for kind_value, color, label in (
        ("gap", "dimgray", "Gap"),
        ("too_short", "darkorange", "Too short"),
    ):
        kind_table = predictions_table.assign(artifact=predictions_table["kind"] == kind_value)
        regions = _artifact_regions(kind_table[["time_s", "artifact"]], start_s, loaded_end_s)
        _shade_regions(fig, regions, color=color, label=label)
        if regions:
            print(f"Shaded {len(regions)} '{kind_value}' region(s) from the predictions parquet")

if ARTIFACT_PATH is not None:
    artifact_table = pd.read_parquet(ARTIFACT_PATH, columns=["time_s", "artifact"])
    regions = _artifact_regions(artifact_table, start_s, loaded_end_s)
    _label_artifacts(fig, regions)
    print(f"Labelled {len(regions)} artifact region(s) from {Path(ARTIFACT_PATH).name}")

fig.suptitle(EDF_PATH.name)

import matplotlib.pyplot as plt

plt.show()


Computing derived signal channels…
Shaded 8 'gap' region(s) from the predictions parquet
Shaded 6 'too_short' region(s) from the predictions parquet


## 7. Wide-Range Overview (Static Hypnogram)

`plot_detailed_comparison`'s state bar draws each state on its own row as a
thin (5pt) line per interval — great for the ~120s windows it's designed for,
but a state that only lasts a few seconds becomes sub-pixel and effectively
invisible once the visible span is more than a few thousand seconds (a
zoom-resolution limit, not a data bug — see the earlier alignment
investigation in this notebook's history).

This cell instead rasterises the *entire* loaded range as one coloured strip:
epochs are binned into at most `MAX_BINS` pixels, and **any bin spanning more
than one state is drawn in `MIXED_COLOR`** rather than silently taking the
majority state — so a brief non-REM/REM excursion always shows up as
something-happened-here, even across a multi-hour recording, instead of
disappearing into whichever state surrounds it. Use it to spot where to zoom
in with the interactive viewer above; it's static (no keyboard navigation).


In [ ]:
from matplotlib.colors import to_rgba


def _hypnogram_overview(
    ax,
    somnotate_vec,
    epoch_s,
    start_s,
    *,
    max_bins=3000,
    state_to_color=None,
    int_to_state=None,
    mixed_color="black",
):
    """Render a wide-range hypnogram as one strip of colour-binned pixels.

    Unlike TimeSeriesStateViewer's per-state line rows, a bin covering more
    than one state is drawn in `mixed_color` instead of picking one -- so a
    state lasting only a few epochs stays visible (as a "something changed
    here" marker) no matter how wide the plotted range is.
    """
    state_to_color = state_to_color or configuration.state_to_color
    int_to_state = int_to_state or configuration.int_to_state

    n_epochs = len(somnotate_vec)
    n_bins = max(1, min(n_epochs, max_bins))
    edges = np.linspace(0, n_epochs, n_bins + 1).astype(int)

    colors = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        unique_values = np.unique(somnotate_vec[lo:hi])
        if len(unique_values) == 1:
            state_name = int_to_state.get(int(unique_values[0]), "undefined")
            colors.append(state_to_color.get(state_name, mixed_color))
        else:
            colors.append(mixed_color)

    image = np.array([to_rgba(color) for color in colors])[np.newaxis, :, :]
    extent = (start_s, start_s + n_epochs * epoch_s, 0, 1)
    ax.imshow(image, aspect="auto", extent=extent, interpolation="nearest")
    ax.set_yticks([])
    ax.set_xlabel("Time [s]")


MAX_BINS = 3000  # roughly one bin per plotted pixel at typical figure widths

fig_overview, ax_overview = plt.subplots(figsize=(14, 1.4))
_hypnogram_overview(ax_overview, somnotate_vec, epoch_s, start_s, max_bins=MAX_BINS)
ax_overview.set_title(f"{EDF_PATH.name} \u2014 state overview ({range_description})")
fig_overview.tight_layout()
plt.show()
